# Comprendre le choix du processus $X_t$

Le virus perturbe le système proie-prédateur,
$$\mathrm{d}V_t = V_t\left(\tfrac23 - \tfrac43 P_t + X_t\right)\mathrm{d}t .$$
On a extrait une réalisation de $(X_t)$ des données. La question est : **quel processus stochastique est $X_t$ ?**

Ce notebook présente les six processus candidats du cours, les explique, et les simule à côté du vrai $X$
pour voir visuellement lequel colle. La règle de lecture est simple, on cherche le processus qui **reste
borné et revient vers une valeur**, comme le vrai $X$.

In [ ]:
import numpy as np
import scipy.stats as scs
import matplotlib.pyplot as plt

## La série à reconnaître

In [ ]:
data = np.loadtxt('virus4.csv', delimiter=',')
V, P = data[:, 0], data[:, 1]
num = P[1:] - P[:-1]; den = P[:-1] * (V[:-1] - 1); ok = np.abs(den) > 1e-3
dt = round(np.median(num[ok] / den[ok]), 3)
X = (V[1:] - V[:-1]) / (V[:-1] * dt) - 2/3 + 4/3 * P[:-1]
n = len(X)
t = np.arange(n) * dt
tau = n * dt
s = 0.052   # echelle de bruit estimee
print('X : plage [%.2f, %.2f], borne et a retour de moyenne' % (X.min(), X.max()))

plt.figure(figsize=(9, 3))
plt.plot(t, X, 'k')
plt.title('Le vrai X a reconnaitre'); plt.xlabel('t'); plt.show()

## 1. Mouvement brownien

$$\mathrm{d}X_t = s\,\mathrm{d}B_t$$

C'est le bruit pur. Sa loi est $X_t \sim \mathcal{N}(X_0,\; s^2 t)$, donc sa variance **croît avec le temps**,
il n'a **aucune borne** et **aucun rappel**. Il s'éloigne indéfiniment de son point de départ.

In [ ]:
rng = np.random.default_rng(1)
plt.figure(figsize=(9, 4))
plt.plot(t, X, 'k', lw=2, label='X reel')
for _ in range(3):
    b = X[0] + np.cumsum(s * np.sqrt(dt) * rng.standard_normal(n))
    plt.plot(t, b, lw=0.8)
plt.title('Mouvement brownien : il part loin, pas de rappel'); plt.xlabel('t'); plt.legend(); plt.show()

**Verdict** non. Il sort largement de la plage du vrai $X$.

## 2. Brownien avec dérive

$$\mathrm{d}X_t = a\,\mathrm{d}t + s\,\mathrm{d}B_t$$

On ajoute une tendance. Sa loi est $X_t \sim \mathcal{N}(X_0 + a t,\; s^2 t)$. Il suit une pente moyenne $a$,
mais la variance s'étale toujours et il n'y a **toujours pas de rappel**.

In [ ]:
a = (X[-1] - X[0]) / tau
plt.figure(figsize=(9, 4))
plt.plot(t, X, 'k', lw=2, label='X reel')
for _ in range(3):
    b = X[0] + a * t + np.cumsum(s * np.sqrt(dt) * rng.standard_normal(n))
    plt.plot(t, b, lw=0.8)
plt.title('Brownien avec derive : tendance + etalement'); plt.xlabel('t'); plt.legend(); plt.show()

**Verdict** non. Il suit la pente mais s'écarte de plus en plus, sans revenir.

## 3. Brownien géométrique (Black-Scholes)

$$\mathrm{d}X_t = \mu X_t\,\mathrm{d}t + \sigma X_t\,\mathrm{d}B_t,
\qquad X_t = X_0\,\exp\!\left(\left(\mu - \tfrac{\sigma^2}{2}\right)t + \sigma B_t\right)$$

Ici le bruit est **proportionnel au niveau** ($\sigma X_t$). Conséquence, le processus reste **toujours du
même signe** et son agitation grossit quand il s'éloigne de zéro.

In [ ]:
S0 = 1.0; mu_g = 0.0; sig_g = 0.3
plt.figure(figsize=(9, 4))
for _ in range(3):
    B = np.cumsum(np.sqrt(dt) * rng.standard_normal(n))
    Sg = S0 * np.exp((mu_g - sig_g**2 / 2) * t + sig_g * B)
    plt.plot(t, Sg, lw=0.9)
plt.title('Brownien geometrique : toujours positif, bruit proportionnel au niveau'); plt.xlabel('t'); plt.show()

**Verdict** non. Notre $X$ est négatif, et son bruit ne grossit pas avec le niveau (on l'a vérifié, la
diffusion est constante). Ce modèle ne correspond pas.

## 4. Pont brownien

$$P_t = B_t - \frac{t}{\tau}\,B_\tau$$

C'est un brownien **forcé de revenir à sa valeur de départ** à l'instant final $t = \tau$. Gaussien,
d'espérance nulle, mais avec cette contrainte d'extrémité.

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(t, X, 'k', lw=2, label='X reel')
for _ in range(3):
    B = np.cumsum(s * np.sqrt(dt) * rng.standard_normal(n))
    pont = X[0] + B - (t / tau) * B[-1]
    plt.plot(t, pont, lw=0.8)
plt.title('Pont brownien : forced a revenir au depart en t=tau'); plt.xlabel('t'); plt.legend(); plt.show()

**Verdict** non. Rien dans l'énoncé n'impose à $X$ de revenir à une valeur précise à la fin.

## 5. Mouvement brownien intégral

$$X_t = \int_0^t B_u\,\mathrm{d}u$$

C'est l'intégrale d'un brownien. Ses trajectoires sont **lisses** ($C^1$), il est gaussien, mais sa variance
vaut $\dfrac{t^3}{3}$, donc elle **explose** encore plus vite que le brownien. Sa dérivée est un brownien.

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(t, X, 'k', lw=2, label='X reel')
for _ in range(3):
    B = np.cumsum(s * np.sqrt(dt) * rng.standard_normal(n))
    I = X[0] + np.cumsum(B) * dt
    plt.plot(t, I, lw=0.8)
plt.title('Brownien integral : lisse mais s etale sans borne'); plt.xlabel('t'); plt.legend(); plt.show()

**Verdict** non. Il est lisse comme $X$, mais il s'étale sans borne, et on a vu que sa dérivée n'est pas
un brownien sur nos données.

## 6. Ornstein-Uhlenbeck (le bon)

$$\mathrm{d}X_t = -\gamma\,(X_t - m)\,\mathrm{d}t + s\,\mathrm{d}B_t$$

Le terme $-\gamma(X_t - m)$ **rappelle** le processus vers la moyenne $m$. Sa loi est
$$X_t \sim \mathcal{N}\!\left(m + (X_0 - m)e^{-\gamma t},\;\; \frac{s^2}{2\gamma}\left(1 - e^{-2\gamma t}\right)\right),$$
avec une **loi stationnaire** $\mathcal{N}\!\left(m,\; \dfrac{s^2}{2\gamma}\right)$. Il reste borné autour de $m$.

In [ ]:
gamma, m = 0.069, -0.657
r = np.exp(-gamma * dt)
e = s * np.sqrt((1 - r**2) / (2 * gamma))
plt.figure(figsize=(9, 4))
plt.plot(t, X, 'k', lw=2, label='X reel')
for _ in range(3):
    o = np.zeros(n); o[0] = X[0]
    for k in range(1, n):
        o[k] = m + r * (o[k-1] - m) + e * rng.standard_normal()
    plt.plot(t, o, lw=0.8)
plt.title('Ornstein-Uhlenbeck : reste borne, revient vers m'); plt.xlabel('t'); plt.legend(); plt.show()

**Verdict** oui. Il reste dans la même bande que le vrai $X$ et revient vers une moyenne, exactement le
comportement observé.

## Synthèse : les quatre questions qui mènent à l'OU

On élimine les candidats par quatre questions auxquelles les données répondent.

1. **$X$ revient-il vers une valeur ?** Oui. Cela élimine le brownien, le brownien avec dérive, le
   géométrique et le brownien intégral, qui tous s'échappent.
2. **Est-ce un retour forcé à la fin ?** Non, donc pas un pont brownien.
3. **Le bruit grossit-il avec le niveau ?** Non, la diffusion est constante, donc pas de géométrique.
4. **Le rappel est-il linéaire ?** Oui en gros, donc un rappel linéaire, c'est l'Ornstein-Uhlenbeck.

Il ne reste qu'un seul survivant,
$$\boxed{\mathrm{d}X_t = -\gamma\,(X_t - m)\,\mathrm{d}t + s\,\mathrm{d}B_t}$$
avec $\gamma \approx 0.07$, $m \approx -0.66$, $s \approx 0.05$. Et comme l'OU est gaussien, on en déduit
directement que $X_\tau$ suit une loi normale, ce qui servira au modèle 2.